<a href="https://colab.research.google.com/github/dkamalakar/rag-demo/blob/feat%2Fdocument_loaders/Document_retriever_search_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies - Langchain etc



In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-huggingface
!pip install jq
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.52
    Uninstalling langchain-core-0.3.52:
      Successfully uninstalled langchain-core-0.3.52
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.23
    Uninstalling langchain-0.3.23:
      Successfully uninstalled langchain-0.3.23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [2]:
!pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.0 MB/s eta 0:00:0

In [3]:
from getpass import getpass

OPENAI_KEY = getpass("Enter your OpenAI Key")

Enter your OpenAI Key··········


In [4]:
import os
os.environ['OPENAI_API_KEY'] = OPENAI_KEY

In [5]:
from langchain_openai import OpenAIEmbeddings
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')


In [6]:
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 15.5MB/s]


In [7]:
!unzip rag_docs.zip

Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  


In [9]:
from langchain.document_loaders import JSONLoader
loader = JSONLoader(
    file_path='./rag_docs/wikidata_rag_demo.jsonl',
    jq_schema='.',
    text_content=False,
    json_lines=True
)

wiki_docs = loader.load()

In [16]:
len(wiki_docs)

1801

In [17]:
wiki_docs[500]

Document(metadata={'source': '/content/rag_docs/wikidata_rag_demo.jsonl', 'seq_num': 501}, page_content='{"id": "99224", "title": "Christopher Lee", "paragraphs": ["Sir Christopher Frank Carandini Lee CBE, CStJ (27 May 1922 7 June 2015) was an English actor. He was a direct descendent of Marie Carandini and Robert E. Lee.", "Lee was best known for his many movie characters. For example, Dracula and Fu Manchu in many movies during the 1950s through the 1970s. He became even better known as Scaramanga in the Bond movie \\"The Man with the Golden Gun\\", Saruman the White in \\"The Lord of the Rings\\" trilogy and Count Dooku in \\"\\".", "He performed roles in 275 movies since 1946. He appeared in over 75 television programs. Lee was also a voice actor, providing his voice in over 50 movies. This made him the Guinness World Record holder for most movie acting roles ever. Lee was also a singer and his latest album was released in 2013."]}')

In [71]:
import json
from langchain.docstore.document import Document
wiki_docs_processed = []

for doc in wiki_docs:
  doc = json.loads(doc.page_content)
  metadata = {
      "title": doc['title'],
      "id": doc['id'],
      "source": 'Wikipedia'
  }

  data = ' '.join(doc['paragraphs'])
  wiki_docs_processed.append(Document(page_content=data,metadata=metadata))

In [72]:
wiki_docs_processed[1500]

Document(metadata={'title': 'Janne Persson', 'id': '460169', 'source': 'Wikipedia'}, page_content='Jan Persson ("Janne Lucas"), born 3 October 1947 in Gothenburg\'s Gamlestad Parish in Gothenburg, Sweden is a Swedish pianist and singer, scoring several chart successes in Sweden during the 1970s and 1980s. Janne Lucas participated at Melodifestivalen 1980 with the song "Växeln hallå", winning the contest. The upcoming year he participated with the song "Rocky Mountain" ending up third. For many years, Janne Lucas also acted as pianist for "Vi i femman" Janne also accompanied the vocal group "Noviserna" for a while, where Anna-Lisa Cederquist participated.')

In [73]:
from langchain.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader("./rag_docs/attention_paper.pdf")
doc_pages = loader.load()

In [74]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=3500,
                                          chunk_overlap=0,


)

doc_chunks = splitter.split_documents(doc_pages)

In [75]:
len(doc_chunks)

16

In [76]:
big_doc = '\n'.join([doc.page_content for doc in doc_chunks])

In [77]:
len(big_doc.split(' '))

5050

In [78]:
from langchain_openai import ChatOpenAI
chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [79]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

def generate_chunk_context(document, chunk):

  chunk_process_prompt = """You are an AI assistant specializing in research paper analysis.
                        Your task is to provide brief, relevant context for a chunk of text based
                        on the following research paper.

                        Here is the research paper:
                        <paper>
                        {paper}
                        </paper>

                        Here is the chunk we want to situate within the whole document.
                        <chunk>
                        {chunk}
                        </chunk>


                        provide a concise context to situate this chunk within the overall document for the
                        purposes of improving search retrieval of the chunk.
                        - Answer only with the succint context and nothing else.
                        - Context should be mentioned like 'Focuses on.....'
                        do not mention 'this chunk or section focuses on...'

                        context:

                        """

  prompt_template = ChatPromptTemplate.from_template(chunk_process_prompt)

  agentic_chunk_chain = (prompt_template
                              |
                         chatgpt
                              |
                         StrOutputParser()


                         )

  context = agentic_chunk_chain.invoke({'paper': document, 'chunk': chunk})

  return context

In [80]:
print(doc_chunks[5].page_content)

output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O
where headi = Attention(QW Q
i , KW K
i , V W V
i )
Where the projections are parameter matrices W Q
i
∈Rdmodel×dk, W K
i
∈Rdmodel×dk, W V
i
∈Rdmodel×dv
and W O ∈Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3
Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and

In [81]:
generate_chunk_context(big_doc, doc_chunks[5].page_content)


'Focuses on the implementation and functionality of multi-head attention in the Transformer model, detailing how it allows for parallel processing of information from different representation subspaces and its application in both encoder-decoder and self-attention layers.'

In [82]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def create_contextual_chunks(file_path):
  print('Loading pages:', file_path)
  loader = PyMuPDFLoader(file_path)
  doc_pages = loader.load()

  print("Chunk pages:", file_path)
  splitter = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=0)
  doc_chunks = splitter.split_documents(doc_pages)

  print("Generating contextual chunks: ", file_path)

  original_doc = '\n'.join([doc.page_content for doc in doc_chunks])
  contextual_chunks = []

  for chunk in doc_chunks:
    context = generate_chunk_context(original_doc, chunk.page_content)
    contextual_chunks.append(Document(page_content=context+'\n'+chunk.page_content,
                                      metadata=chunk.metadata))

  print("Finished Processing : ", file_path)
  print()
  return contextual_chunks



In [83]:
from glob import glob
pdf_files = glob('./rag_docs/*.pdf')
pdf_files



['./rag_docs/resnet_paper.pdf',
 './rag_docs/cnn_paper.pdf',
 './rag_docs/attention_paper.pdf',
 './rag_docs/vision_transformer.pdf']

In [84]:
paper_docs = []
for fp in pdf_files:
  paper_docs.extend(create_contextual_chunks(fp))

Loading pages: ./rag_docs/resnet_paper.pdf
Chunk pages: ./rag_docs/resnet_paper.pdf
Generating contextual chunks:  ./rag_docs/resnet_paper.pdf
Finished Processing :  ./rag_docs/resnet_paper.pdf

Loading pages: ./rag_docs/cnn_paper.pdf
Chunk pages: ./rag_docs/cnn_paper.pdf
Generating contextual chunks:  ./rag_docs/cnn_paper.pdf
Finished Processing :  ./rag_docs/cnn_paper.pdf

Loading pages: ./rag_docs/attention_paper.pdf
Chunk pages: ./rag_docs/attention_paper.pdf
Generating contextual chunks:  ./rag_docs/attention_paper.pdf
Finished Processing :  ./rag_docs/attention_paper.pdf

Loading pages: ./rag_docs/vision_transformer.pdf
Chunk pages: ./rag_docs/vision_transformer.pdf
Generating contextual chunks:  ./rag_docs/vision_transformer.pdf
Finished Processing :  ./rag_docs/vision_transformer.pdf



In [85]:
len(paper_docs)

79

In [86]:
paper_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'source': './rag_docs/resnet_paper.pdf', 'file_path': './rag_docs/resnet_paper.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'trapped': '', 'modDate': 'D:20151211011345Z', 'creationDate': 'D:20151211011345Z', 'page': 0}, page_content='Focuses on the introduction of a residual learning framework to facilitate the training of deeper neural networks, addressing challenges such as vanishing gradients and degradation of accuracy with increased depth. It highlights the empirical success of residual networks on the ImageNet dataset and their application in various visual recognition tasks, including significant improvements in object detection.\nDeep Residual Learning for Image Recognition\nKaiming He\nXiangyu Zhang\nShaoqing Ren\nJian Sun\nMicrosoft Research\n{k

In [87]:
len(wiki_docs_processed)
total_docs = wiki_docs_processed + paper_docs
len(total_docs)

1880

In [88]:
from langchain_chroma import Chroma

chroma_db = Chroma.from_documents(documents=total_docs,
                                  collection_name='my_db',
                                  embedding=openai_embed_model,
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./my_db")

In [49]:
#Load from disk
chroma_db = Chroma(persist_directory="./my_db",
                   collection_name='my_db',
                   embedding_function=openai_embed_model)

In [62]:
chroma_db

In [89]:
similarity_retriever = chroma_db.as_retriever(search_type='similarity',
                                              search_kwargs={"k": 5})

In [90]:
from IPython.display import display, Markdown

def display_docs(docs):
  for doc in docs:
    print("metadata: ", doc.metadata)
    print("Content Brief: ")
    display(Markdown(doc.page_content[:1000]))
    print()

In [91]:
query = "What is Machine learning ?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

metadata:  {}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {'id': '564928', 'source': 'Wikipedia', 'title': 'Machine learning'}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.

In [92]:
top_docs

[Document(id='686b23d0-8241-4003-900f-941c61185643', metadata={}, page_content='Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.'),
 Document(id='85920ad5-ad5a-4ae8-8aff-8300ba2ce4b5', metadata={}, page_content='Machine learning gives computers the ability to learn without being explicitly programmed (Ar

In [93]:
query = "What is ML ?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

metadata:  {}
Content Brief: 


Standard ML is a functional programming language which is a dialect of ML (programming language). It is sometimes used for writing compilers and in theorem provers. Here is an example of a factorial function written in a simple, non-tail recursive, style.


metadata:  {}
Content Brief: 


Standard ML is a functional programming language which is a dialect of ML (programming language). It is sometimes used for writing compilers and in theorem provers. Here is an example of a factorial function written in a simple, non-tail recursive, style.


metadata:  {}
Content Brief: 


Standard ML is a functional programming language which is a dialect of ML (programming language). It is sometimes used for writing compilers and in theorem provers. Here is an example of a factorial function written in a simple, non-tail recursive, style.


metadata:  {'id': '312307', 'source': 'Wikipedia', 'title': 'Standard ML'}
Content Brief: 


Standard ML is a functional programming language which is a dialect of ML (programming language). It is sometimes used for writing compilers and in theorem provers. Here is an example of a factorial function written in a simple, non-tail recursive, style.


metadata:  {'id': '564928', 'source': 'Wikipedia', 'title': 'Machine learning'}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.

In [94]:
query = "What is difference between transformers and vision transformers ?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

metadata:  {'author': '', 'creationDate': 'D:20210604001958Z', 'creationdate': '2021-06-04T00:19:58+00:00', 'creator': 'LaTeX with hyperref', 'file_path': './rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'moddate': '2021-06-04T00:19:58+00:00', 'page': 7, 'producer': 'pdfTeX-1.40.21', 'source': './rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief: 


Focuses on a controlled scaling study of various models, comparing the transfer performance of Vision Transformers and ResNets when pre-trained on the JFT-300M dataset, and analyzing the performance versus pre-training compute. It highlights the efficiency of Vision Transformers in terms of compute requirements and discusses the implications for future scaling efforts.
Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plus
L/16 and H/14 pre-trained for 14 epochs; and 5 hybrids, R50+ViT-B/32, B/16, L/32, L/16 pr


metadata:  {'author': '', 'creationDate': 'D:20210604001958Z', 'creationdate': '2021-06-04T00:19:58+00:00', 'creator': 'LaTeX with hyperref', 'file_path': './rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'moddate': '2021-06-04T00:19:58+00:00', 'page': 7, 'producer': 'pdfTeX-1.40.21', 'source': './rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief: 


Focuses on a controlled scaling study of various models, comparing the transfer performance of Vision Transformers and ResNets when pre-trained on the JFT-300M dataset, and analyzing the performance versus pre-training compute. It highlights the efficiency of Vision Transformers in terms of compute requirements and discusses the implications for future scaling efforts.
Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plus
L/16 and H/14 pre-trained for 14 epochs; and 5 hybrids, R50+ViT-B/32, B/16, L/32, L/16 pr


metadata:  {'author': '', 'creationDate': 'D:20210604001958Z', 'creationdate': '2021-06-04T00:19:58+00:00', 'creator': 'LaTeX with hyperref', 'file_path': './rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'moddate': '2021-06-04T00:19:58+00:00', 'page': 7, 'producer': 'pdfTeX-1.40.21', 'source': './rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief: 


Focuses on a controlled scaling study of various models, comparing the transfer performance of Vision Transformers and ResNets when pre-trained on the JFT-300M dataset, and analyzing the performance versus pre-training compute. It highlights the efficiency of Vision Transformers in terms of compute requirements and discusses the implications for future scaling efforts.
Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plus
L/16 and H/14 pre-trained for 14 epochs; and 5 hybrids, R50+ViT-B/32, B/16, L/32, L/16 pr


metadata:  {'author': '', 'creationDate': 'D:20210604001958Z', 'creationdate': '2021-06-04T00:19:58+00:00', 'creator': 'LaTeX with hyperref', 'file_path': './rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'moddate': '2021-06-04T00:19:58+00:00', 'page': 0, 'producer': 'pdfTeX-1.40.21', 'source': './rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief: 


Focuses on the introduction of the Vision Transformer (ViT) model, highlighting its ability to perform image classification tasks by applying a standard Transformer architecture directly to sequences of image patches, and contrasting its performance with traditional convolutional neural networks (CNNs) on various benchmarks.
Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16X16 WORDS:
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution, †equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language processing tasks, its applications to computer vision remain limited. In
vision, attention is either applied in conjunction wit


metadata:  {'author': '', 'creationDate': 'D:20210604001958Z', 'creationdate': '2021-06-04T00:19:58+00:00', 'creator': 'LaTeX with hyperref', 'file_path': './rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'moddate': '2021-06-04T00:19:58+00:00', 'page': 0, 'producer': 'pdfTeX-1.40.21', 'source': './rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief: 


Focuses on the introduction of the Vision Transformer (ViT) model, highlighting its ability to perform image classification tasks by applying a standard Transformer architecture directly to sequences of image patches, and contrasting its performance with traditional convolutional neural networks (CNNs) on various benchmarks.
Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16X16 WORDS:
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution, †equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language processing tasks, its applications to computer vision remain limited. In
vision, attention is either applied in conjunction wit

In [95]:
from langchain_openai import ChatOpenAI
chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
from langchain.retrievers.multi_query import MultiQueryRetriever

import logging

similarity_retriever = chroma_db.as_retriever(search_type='similarity',
                                              search_kwargs={"k": 5})

mq_retriever = MultiQueryRetriever.from_llm(retriever=similarity_retriever,
                                            llm=chatgpt)

logging.basicConfig()
logging.getLogger("langchain.retrievers.multiquery").setLevel(logging.INFO)


In [98]:
query="what is cnn ?"
top_docs = mq_retriever.invoke(query)
display_docs(top_docs)

metadata:  {}
Content Brief: 


The Cable News Network (CNN) is an American cable news television channel. It was founded in 1980 by Ted Turner. The Cable News Network first aired on television on June 1, 1980. The Cable News Network's first newscast was anchored (hosted) by David Walker and his wife Lois Hart. In its first year CNN hired many political analysts, including Rowland Evans and Robert Novak. On January 1, 1982 CNN launched a 24-hour sister newscast channel with no talk shows or commentary shows called CNN2. CNN broadcasts programs from its headquarters at the CNN Center in Atlanta, or from the Time Warner Center in New York City, or from studios in Washington, D.C., and Los Angeles. CNN is owned by Time Warner, and the U.S. news channel is a part of the Turner Broadcasting System. The hosts of its opinion shows are Don Lemon, Chris Cuomo, Fredricka Whitfield, Erin Burnett, Brianna Keiler and Brooke Baldwin. CNN has been criticized by the right-wing Media Research Center for having a left-wing bias. Accor


metadata:  {}
Content Brief: 


The Cable News Network (CNN) is an American cable news television channel. It was founded in 1980 by Ted Turner. The Cable News Network first aired on television on June 1, 1980. The Cable News Network's first newscast was anchored (hosted) by David Walker and his wife Lois Hart. In its first year CNN hired many political analysts, including Rowland Evans and Robert Novak. On January 1, 1982 CNN launched a 24-hour sister newscast channel with no talk shows or commentary shows called CNN2. CNN broadcasts programs from its headquarters at the CNN Center in Atlanta, or from the Time Warner Center in New York City, or from studios in Washington, D.C., and Los Angeles. CNN is owned by Time Warner, and the U.S. news channel is a part of the Turner Broadcasting System. The hosts of its opinion shows are Don Lemon, Chris Cuomo, Fredricka Whitfield, Erin Burnett, Brianna Keiler and Brooke Baldwin. CNN has been criticized by the right-wing Media Research Center for having a left-wing bias. Accor


metadata:  {}
Content Brief: 


...

Focuses on the limitations of traditional artificial neural networks (ANNs) in handling image data, particularly regarding computational complexity and the issue of overfitting, while introducing the architecture of convolutional neural networks (CNNs) designed specifically for image processing tasks.
Introduction to Convolutional Neural Networks
3
more suited for image-focused tasks - whilst further reducing the parameters
required to set up the model.
One of the largest limitations of traditional forms of ANN is that they tend to
struggle with the computational complexity required to compute image data.
Common machine learning benchmarking datasets such as the MNIST database
of handwritten digits are suitable for most forms of ANN, due to its relatively
small image dimensionality of just 28 × 28. With this dataset a single neuron in
the ﬁrst hidden layer will contain 784 weights (28×28×1 where 1 bare in mind
that MNIST is normalised to just black and white values), which is manageabl


metadata:  {'author': '', 'creationDate': 'D:20151203014807Z', 'creationdate': '2015-12-03T01:48:07+00:00', 'creator': 'LaTeX with hyperref package', 'file_path': './rag_docs/cnn_paper.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20151203014807Z', 'moddate': '2015-12-03T01:48:07+00:00', 'page': 0, 'producer': 'pdfTeX-1.40.12', 'source': './rag_docs/cnn_paper.pdf', 'subject': '', 'title': '', 'total_pages': 11, 'trapped': ''}
Content Brief: 


Focuses on the introduction of Convolutional Neural Networks (CNNs) within the broader field of Artificial Neural Networks (ANNs), highlighting their significance in image-driven pattern recognition tasks and providing an overview of their architecture and learning processes.
An Introduction to Convolutional Neural Networks
Keiron O’Shea1 and Ryan Nash2
1 Department of Computer Science, Aberystwyth University, Ceredigion, SY23 3DB
keo7@aber.ac.uk
2 School of Computing and Communications, Lancaster University, Lancashire, LA1
4YW
nashrd@live.lancs.ac.uk
Abstract. The ﬁeld of machine learning has taken a dramatic twist in re-
cent times, with the rise of the Artiﬁcial Neural Network (ANN). These
biologically inspired computational models are able to far exceed the per-
formance of previous forms of artiﬁcial intelligence in common machine
learning tasks. One of the most impressive forms of ANN architecture is
that of the Convolutional Neural Network (CNN). CNNs are primarily
used to solv


metadata:  {'id': '14059', 'source': 'Wikipedia', 'title': 'News'}
Content Brief: 


News is when people talk about current events (things that are happening right now). News Media is a portrayal of current affairs, perspectives and social influence. News can be given in newspapers, television, magazines, or radio. There are several news channels on cable television that give news all day long, such as Fox News and CNN. There are several news magazines, such as "Time", "The Economist", and "Newsweek". A newsman is a person who helps out with the news. For example, Brian Gotter is a newsman. News Media can be viewed in many forms, such as newspaper, television and radio.

In [99]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor


compressor = LLMChainExtractor.from_llm(llm=chatgpt)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=mq_retriever
)


In [100]:
query = "What is ML"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

metadata:  {}
Content Brief: 


Standard ML is a functional programming language which is a dialect of ML (programming language).


metadata:  {}
Content Brief: 


Standard ML is a functional programming language which is a dialect of ML (programming language).


metadata:  {}
Content Brief: 


...

Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.


metadata:  {}
Content Brief: 


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised.

In [101]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainFilter

_filter = LLMChainFilter.from_llm(llm=chatgpt)


# compressor = LLMChainExtractor.from_llm(llm=chatgpt)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=mq_retriever
)


In [102]:
query = "What is ML"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

metadata:  {}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


...

Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


metadata:  {}
Content Brief: 


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.


metadata:  {}
Content Brief: 


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer. Certain tasks, such as as recognizing and understanding speech, images or handwriting, is easy to do for humans. However, for a computer, these tasks are very difficult to do. In a multi-layer neural network (having more than two layers), the information processed will become more abstract with each added layer. Deep learning models are inspired by information processing and communication patterns in biological nervous systems; they are different from the structural and functional properties of biological brains (especially the human brain) in many ways, whic

In [103]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

similarity_retriever = chroma_db.as_retriever(search_type='similarity', search_kwargs={'k': 5})

_filter = LLMChainFilter.from_llm(llm=chatgpt)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=similarity_retriever
)

reranker = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-large')
reranker_compressor = CrossEncoderReranker(model=reranker, top_n=3)

final_retriever = ContextualCompressionRetriever(
    base_compressor=reranker_compressor, base_retriever=compression_retriever
)

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [104]:
query = "what is ML ?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

metadata:  {}
Content Brief: 


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.